# 📍 GBFS Station Reference Ingestion

Fetches live station metadata from the official **Bike Share Toronto GBFS feed** and persists it to Bronze and Silver layers.

**User Story:** US09 – Data Preprocessing

| Layer | Path | Description |
|---|---|---|
| **Bronze** | `data/bronze/station_id` | Raw station data as retrieved from GBFS |
| **Silver** | `data/silver/station_id` | Clean station dimension table for downstream joins |

| Step | Purpose |
|---|---|
| 1 | Define Bronze/Silver paths |
| 2 | Fetch station reference data from GBFS API |
| 3 | Create structured Spark DataFrame with explicit schema |
| 4 | Run data quality checks (nulls, duplicates) |
| 5 | Write Bronze and Silver Parquet |
| 6 | Validate Silver row count |

## Step 1 — Configuration

Defines the DBFS output paths for the Bronze and Silver layers. Both paths point to the Unity Catalog Volume used across all Capstone pipeline notebooks.


In [0]:
# =========================
## STEP 1 - CONFIG
# =========================

BRONZE_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/bronze/station_id"
SILVER_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/silver/station_id"

#BRONZE_DIR = "dbfs:/Volumes/default/bike_share/Projects/Capstone/data/bronze/station_id"
#SILVER_DIR = "dbfs:/Volumes/default/bike_share/Projects/Capstone/data/silver/station_id"

print("BRONZE_DIR =", BRONZE_DIR)
print("SILVER_DIR =", SILVER_DIR)


## Step 2 — Extract Station Reference Data from GBFS

Fetches live station metadata from the Bike Share Toronto GBFS feed using a two-step API call:

1. **GBFS root feed** (`gbfs.json`) is queried to discover all available endpoint URLs dynamically — this avoids hardcoding URLs that may change over time.
2. The `station_information` endpoint URL is located from the feed list and fetched to retrieve the full list of active stations.

The result is a list of station dictionaries containing `station_id`, `name`, `lat`, and `lon` among other fields. The total number of stations retrieved is printed as a quick sanity check.


In [0]:
# =========================
## STEP 2 – Extract Station Reference Data
# =========================

import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# --- Reusable function (from US06 style) ---
def fetch_gbfs_json(url: str) -> dict:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

# GBFS root used in US06
gbfs_root_url = "https://toronto-us.publicbikesystem.net/customer/gbfs/v2/gbfs.json"

# Get GBFS root
gbfs_root = fetch_gbfs_json(gbfs_root_url)

# Locate station_information endpoint dynamically
feeds = gbfs_root["data"]["en"]["feeds"]

station_info_url = None
for feed in feeds:
    if feed.get("name") == "station_information":
        station_info_url = feed.get("url")
        break

if not station_info_url:
    raise Exception("station_information feed not found.")

print("station_information URL:", station_info_url)

# Fetch station_information JSON
station_info_json = fetch_gbfs_json(station_info_url)
stations = station_info_json["data"]["stations"]

print("Number of stations retrieved:", len(stations))




## Step 3 — Create Structured Spark DataFrame

Converts the raw station list into a Spark DataFrame with an **explicit schema** (`station_id`, `name`, `lat`, `lon`). Only these four fields are retained to keep the dimension table minimal and avoid type inference issues from the richer GBFS payload.

Each field is cast explicitly:
- `station_id` → `StringType` (some IDs may be numeric strings)
- `lat` / `lon` → `DoubleType`
- Missing values are safely set to `None` rather than causing a parse error.


In [0]:
# =========================
#STEP 3 – Create Structured DataFrame
# =========================

station_schema = StructType([
    StructField("station_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
])

# Keep only required fields (avoid type inference issues)
stations_min = [
    {
        "station_id": str(s.get("station_id")) if s.get("station_id") is not None else None,
        "name": s.get("name"),
        "lat": float(s.get("lat")) if s.get("lat") is not None else None,
        "lon": float(s.get("lon")) if s.get("lon") is not None else None,
    }
    for s in stations
]

df_station_info = spark.createDataFrame(stations_min, schema=station_schema)

print("Rows in DataFrame:", df_station_info.count())
df_station_info.printSchema()

display(df_station_info.limit(10))

## Step 4 — Data Quality Checks

Validates the station DataFrame across three dimensions before writing:

| Check | What it catches |
|---|---|
| **Null `station_id`** | Stations with no identifier — unusable for joins |
| **Null `lat` or `lon`** | Stations with missing coordinates — unusable for spatial features |
| **Duplicate `station_id`** | Same station appearing more than once in the feed |

All three counts should be **zero** for a clean dataset. Any non-zero result indicates a data quality issue in the upstream GBFS feed that would need to be investigated before proceeding.


In [0]:
# =========================
#STEP 4 – Data Quality Checks
# =========================

from pyspark.sql import functions as F

total_rows = df_station_info.count()

null_station_id = df_station_info.filter(F.col("station_id").isNull()).count()
null_lat_lon = df_station_info.filter(
    F.col("lat").isNull() | F.col("lon").isNull()
).count()

duplicate_station_id = (
    df_station_info
    .groupBy("station_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total rows:", total_rows)
print("Null station_id:", null_station_id)
print("Null lat or lon:", null_lat_lon)
print("Duplicate station_id:", duplicate_station_id)

## Step 5 — Write Bronze and Silver

Writes the station data to both layers as single-file Parquet (via `coalesce(1)`) since the station reference table is small (~1,000 rows) and does not benefit from partitioning:

- **Bronze** — written directly from the Spark DataFrame, preserving the raw-faithful copy.
- **Silver** — read back from Bronze and written again as the dimension table. Reading from Bronze before writing Silver ensures the Silver copy exactly matches what was persisted, not just what was in memory.

The DBFS directory contents are listed after each write to confirm the Parquet file was created.


In [0]:
# ============================
# STEP 5 - Write Bronze and Silver (single file, no partition)
# ============================

# Force single file (dataset is small)
df_station_single = df_station_info.coalesce(1)

# Write Bronze (no partition)
df_station_single.write.mode("overwrite").parquet(BRONZE_DIR)
print("Bronze write completed (single file).")

# Read back Bronze (good practice to ensure consistency)
df_bronze = spark.read.parquet(BRONZE_DIR)

# Write Silver (dimension table copy)
df_bronze.coalesce(1).write.mode("overwrite").parquet(SILVER_DIR)
print("Silver write completed (single file).")

# Verify structure
print("Bronze path content:")
display(dbutils.fs.ls(BRONZE_DIR))

print("Silver path content:")
display(dbutils.fs.ls(SILVER_DIR))

## Step 6 — Validate Silver Row Count

Reads the Silver Parquet back from DBFS and returns the row count as a final confirmation that the write completed successfully and no rows were lost. The count should match the number of stations retrieved in Step 2.


In [0]:
spark.read.parquet(SILVER_DIR).count()